In [67]:
import pandas as pd
import numpy as np
import pandas_ta as ta

# CSVファイルを読み込む
df = pd.read_csv(
    'data/AAPL.csv',
    parse_dates=['Date'],
    date_format='%m/%d/%Y',
    index_col='Date'
)

# データを日付順にソート
df = df.sort_index()

In [68]:
# 必要な列を選択
ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
# 両方のリターンを計算
# df_pct = df[ohlcv_cols].pct_change()
df_log = np.log(df[ohlcv_cols] / df[ohlcv_cols].shift(1))
df_log * 100

,Open,High,Low,Close,Volume
Date,,,,,
2015-02-26,NaN,NaN,NaN,NaN,NaN
2015-02-27,0.939166,-0.229498,1.279201,-1.510257,-38.671945
2015-03-02,-0.578594,-0.222350,0.046776,0.489226,-25.393673
2015-03-03,-0.224623,-0.585067,-0.163813,0.208938,-24.110696
2015-03-04,0.108502,0.030878,0.179400,-0.635908,-17.819033
...,...,...,...,...,...
2025-02-19,0.208670,0.337955,0.544496,0.163486,-41.609357
2025-02-20,0.114379,0.312507,0.463474,0.391278,0.349312
2025-02-21,0.411498,0.770989,0.379972,-0.113965,49.841946


In [69]:
df_log_open_volume = np.log(df[['Open', 'Volume']] / df[['Open', 'Volume']].shift(1)) * 10000
df_log_open_volume = df_log_open_volume.rename(columns={'Open': 'OpenOfs', 'Volume': 'VolumeOfs'})
# 一度に計算する方法
cols = ['High', 'Low', 'Close']
# 対数リターンの場合
df_log_from_open = pd.DataFrame({
        f"{col}_From_Open": np.log(df[col] / df['Open']) * 10000
        for col in cols
    },
    index=df.index
)
df_ofs = pd.concat([df_log_open_volume, df_log_from_open], axis=1)
df_ofs[1:]


,OpenOfs,VolumeOfs,High_From_Open,Low_From_Open,Close_From_Open
Date,,,,,
2015-02-27,93.916562,-3867.194543,43.750310,-136.309422,-119.168786
2015-03-02,-57.859371,-2539.367310,79.374670,-73.772418,-12.386779
2015-03-03,-22.462346,-2411.069558,43.330306,-67.691369,30.969365
2015-03-04,10.850191,-1781.903295,35.567965,-60.601537,-43.471579
2015-03-05,-40.360191,5795.220395,13.212608,-221.759500,-170.206862
...,...,...,...,...,...
2025-02-19,20.867011,-4160.935675,55.026939,-61.481837,8.579659
2025-02-20,11.437910,34.931228,74.839689,-26.572384,36.269575
2025-02-21,41.149806,4984.194643,110.788777,-29.724964,-16.276708


In [70]:
# カスタム戦略を作成
my_strategy = ta.Strategy(
    name="基本指標",
    ta=[
        # trend
        {"kind": "sma", "length": 5},
        {"kind": "sma", "length": 25},
        {"kind": "sma", "length": 75},
        # momentum
        {"kind": "rsi"},
        {"kind": "roc"},
        # volatility
        {"kind": "atr"},
        {"kind": "bbands"},
        # volume
        {"kind": "obv"},
        {"kind": "vwap"},
    ]
)

# 戦略を適用
df.ta.strategy(my_strategy)
df[75:]

,Close,Volume,Open,High,Low,SMA_5,SMA_25,SMA_75,RSI_14,ROC_10,ATRr_14,BBL_5_2.0,BBM_5_2.0,BBU_5_2.0,BBB_5_2.0,BBP_5_2.0,OBV,VWAP_D
Date,,,,,,,,,,,,,,,,,,
2015-06-15,31.730,175587840,31.5250,31.8100,31.4275,31.9490,32.292944,31.878565,42.861661,-2.769223,0.479244,31.554911,31.9490,32.343089,2.466987,0.222144,-1.623975e+09,31.655833
2015-06-16,31.900,125774600,31.7575,31.9625,31.5925,31.9580,32.305744,31.875699,45.793777,-1.815943,0.471413,31.570916,31.9580,32.345084,2.422454,0.425081,-1.498200e+09,31.818333
2015-06-17,31.825,131435600,31.9300,31.9700,31.6850,31.8790,32.320096,31.869732,44.703855,-2.167230,0.458053,31.588990,31.8790,32.169010,1.819444,0.406900,-1.629636e+09,31.826667
2015-06-18,31.970,141455960,31.8075,32.0775,31.8050,31.8435,32.338796,31.864799,47.314636,-1.144094,0.444758,31.676120,31.8435,32.010880,1.051265,0.877883,-1.488180e+09,31.950833
2015-06-19,31.650,217446120,31.9275,31.9550,31.6000,31.8150,32.315296,31.858332,42.540994,-1.593471,0.439403,31.585566,31.8150,32.044434,1.442301,0.140420,-1.705626e+09,31.735000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-19,244.870,32204220,244.6600,246.0100,243.1604,242.4680,233.204400,236.886400,60.668817,5.184708,5.675701,236.366166,242.4680,248.569834,5.033105,0.696826,2.396930e+09,244.680133
2025-02-20,245.830,32316910,244.9400,246.7800,244.2900,244.2600,233.706400,237.096133,61.640571,5.746978,5.448151,241.368689,244.2600,247.151311,2.367405,0.771503,2.429247e+09,245.633333
2025-02-21,245.550,53197430,245.9500,248.6900,245.2200,245.0640,234.013600,237.358000,61.165892,5.286854,5.306854,243.994899,245.0640,246.133101,0.872507,0.727294,2.376050e+09,246.486667


In [71]:
# SMA5を基準とした差分計算を追加
df['SMA_25_pct'] = (df['SMA_25'] - df['SMA_5']) / df['SMA_5'] * 100
df['SMA_75_pct'] = (df['SMA_75'] - df['SMA_5']) / df['SMA_5'] * 100
df

,Close,Volume,Open,High,Low,SMA_5,SMA_25,SMA_75,RSI_14,ROC_10,ATRr_14,BBL_5_2.0,BBM_5_2.0,BBU_5_2.0,BBB_5_2.0,BBP_5_2.0,OBV,VWAP_D,SMA_25_pct,SMA_75_pct
Date,,,,,,,,,,,,,,,,,,,,
2015-02-26,32.6037,364687320,32.1962,32.7175,31.6525,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.646873e+08,32.324567,NaN,NaN
2015-02-27,32.1150,247725400,32.5000,32.6425,32.0600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.169619e+08,32.272500,NaN,NaN
2015-03-02,32.2725,192170720,32.3125,32.5700,32.0750,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.091326e+08,32.305833,NaN,NaN
2015-03-03,32.3400,150999600,32.2400,32.3800,32.0225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.601322e+08,32.247500,NaN,NaN
2015-03-04,32.1350,126353920,32.2750,32.3900,32.0800,32.29324,NaN,NaN,NaN,NaN,NaN,31.940183,32.29324,32.646297,2.186568,0.275900,3.337783e+08,32.201667,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-19,244.8700,32204220,244.6600,246.0100,243.1604,242.46800,233.2044,236.886400,60.668817,5.184708,5.675701,236.366166,242.46800,248.569834,5.033105,0.696826,2.396930e+09,244.680133,-3.820545,-2.301994
2025-02-20,245.8300,32316910,244.9400,246.7800,244.2900,244.26000,233.7064,237.096133,61.640571,5.746978,5.448151,241.368689,244.26000,247.151311,2.367405,0.771503,2.429247e+09,245.633333,-4.320642,-2.932886
2025-02-21,245.5500,53197430,245.9500,248.6900,245.2200,245.06400,234.0136,237.358000,61.165892,5.286854,5.306854,243.994899,245.06400,246.133101,0.872507,0.727294,2.376050e+09,246.486667,-4.509189,-3.144485


In [72]:
df['SMA_25_logdiff'] = np.log(df['SMA_25'] / df['SMA_5'])
df['SMA_75_logdiff'] = np.log(df['SMA_75'] / df['SMA_5'])
df * 100

,Close,Volume,Open,High,Low,SMA_5,SMA_25,SMA_75,RSI_14,ROC_10,...,BBM_5_2.0,BBU_5_2.0,BBB_5_2.0,BBP_5_2.0,OBV,VWAP_D,SMA_25_pct,SMA_75_pct,SMA_25_logdiff,SMA_75_logdiff
Date,,,,,,,,,,,,,,,,,,,,,
2015-02-26,3260.37,36468732000,3219.62,3271.75,3165.25,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.646873e+10,3232.456667,NaN,NaN,NaN,NaN
2015-02-27,3211.50,24772540000,3250.00,3264.25,3206.00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.169619e+10,3227.250000,NaN,NaN,NaN,NaN
2015-03-02,3227.25,19217072000,3231.25,3257.00,3207.50,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.091326e+10,3230.583333,NaN,NaN,NaN,NaN
2015-03-03,3234.00,15099960000,3224.00,3238.00,3202.25,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4.601322e+10,3224.750000,NaN,NaN,NaN,NaN
2015-03-04,3213.50,12635392000,3227.50,3239.00,3208.00,3229.324,NaN,NaN,NaN,NaN,...,3229.324,3264.629688,218.656834,27.590013,3.337783e+10,3220.166667,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-19,24487.00,3220422000,24466.00,24601.00,24316.04,24246.800,23320.44,23688.640000,6066.881675,518.470790,...,24246.800,24856.983448,503.310497,69.682605,2.396930e+11,24468.013333,-382.054539,-230.199449,-3.895442,-2.328904
2025-02-20,24583.00,3231691000,24494.00,24678.00,24429.00,24426.000,23370.64,23709.613333,6164.057103,574.697810,...,24426.000,24715.131112,236.740450,77.150312,2.429247e+11,24563.333333,-432.064194,-293.288572,-4.416761,-2.976755
2025-02-21,24555.00,5319743000,24595.00,24869.00,24522.00,24506.400,23401.36,23735.800000,6116.589178,528.685361,...,24506.400,24613.310056,87.250723,72.729387,2.376050e+11,24648.666667,-450.918944,-314.448471,-4.614017,-3.194985
